# DINOv2 vs ResNet50 vs CLIP trên SkinCancerMNIST


In [ ]:
import os
# Đưa đường dẫn làm việc (Current Working Directory) ra ngoài thư mục gốc của project
# Để đọc đúng thư mục data/, logs/ và weights/
if os.path.basename(os.getcwd()) == "skin":
    os.chdir("../../")


In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

# Ánh xạ 7 loại bệnh của bộ dữ liệu HAM10000 thành các số nguyên (index) để model dễ học
DX_CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_TO_IDX = {cls: idx for idx, cls in enumerate(DX_CLASSES)}
IDX_TO_CLASS = {idx: cls for cls, idx in CLASS_TO_IDX.items()}

class SkinCancerDataset(Dataset):
    """
    Lớp Dataset tùy chỉnh cho bộ dữ liệu SkinCancerMNIST (HAM10000).
    """
    def __init__(self, df, img_dirs, transform=None):
        """
        Khởi tạo Dataset.
        Args:
            df (pd.DataFrame): Bảng dữ liệu pandas chứa cột 'image_id' và 'dx' (nhãn bệnh).
            img_dirs (list): Danh sách các đường dẫn chứa thư mục ảnh (vì HAM10000 chia ảnh ra 2 phần).
            transform (callable, optional): Các phép biến đổi hình ảnh (ví dụ: resize, crop, normalize).
        """
        self.df = df
        self.img_dirs = img_dirs
        self.transform = transform

    def __len__(self):
        # Trả về tổng số lượng ảnh trong tập dữ liệu
        return len(self.df)

    def _get_image_path(self, image_id):
        # Hàm hỗ trợ tìm kiếm ảnh xem nó nằm ở thư mục part_1 hay part_2
        img_name = f"{image_id}.jpg"
        for d in self.img_dirs:
            p = os.path.join(d, img_name)
            if os.path.exists(p):
                return p
        raise FileNotFoundError(f"Không tìm thấy ảnh {img_name} trong các thư mục được cung cấp.")

    def __getitem__(self, idx):
        # Lấy ra 1 sample (ảnh và nhãn) tại vị trí idx
        row = self.df.iloc[idx]
        image_id = row['image_id']
        label = CLASS_TO_IDX[row['dx']] # Chuyển đổi nhãn chuỗi thành số nguyên
        
        # Đọc ảnh và chuyển thành định dạng RGB
        img_path = self._get_image_path(image_id)
        image = Image.open(img_path).convert("RGB")
        
        # Áp dụng các phép biến đổi (nếu có)
        if self.transform:
            image = self.transform(image)
            
        return image, label


def get_transforms(model_type="dinov2", is_train=True):
    """
    Tạo các phép biến đổi (transforms) hình ảnh phù hợp với từng loại mô hình.
    Việc chuẩn hóa dữ liệu phải giống hệt lúc mô hình được Pre-train.
    """
    if model_type == "dinov2" or model_type == "resnet50":
        # DINOv2 và ResNet50 đều sử dụng chuẩn hóa của ImageNet
        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]
        target_size = 224
        
        if is_train:
            # Data augmentation cho tập Train để tránh overfitting
            return transforms.Compose([
                transforms.RandomResizedCrop(target_size, scale=(0.8, 1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std)
            ])
        else:
            # Biến đổi cơ bản cho tập Validation / Test (không làm sai lệch dữ liệu)
            return transforms.Compose([
                transforms.Resize(256),
                transforms.CenterCrop(target_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=mean, std=std)
            ])
    elif model_type == "clip":
        import open_clip
        # OpenCLIP cung cấp sẵn transform đi kèm với trọng số, ta chỉ việc lấy ra dùng
        _, train_transform, val_transform = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
        return train_transform if is_train else val_transform
    else:
        raise ValueError(f"Không hỗ trợ mô hình: {model_type}")

def get_dataloaders(csv_path, img_dirs, model_type="dinov2", batch_size=32, num_workers=4, test_size=0.2, val_size=0.1):
    """
    Hàm đọc file CSV và chia dữ liệu thành 3 tập: Train, Val, Test.
    Sau đó đóng gói thành các đối tượng DataLoader để Pytorch có thể đọc theo từng Batch.
    """
    # Đọc file CSV metadata
    df = pd.read_csv(csv_path)
    
    # Bước 1: Tách tập Test (20%) ra khỏi toàn bộ dữ liệu. Giữ lại Train+Val (80%)
    # stratify=df['dx'] giúp phân bố các nhãn bệnh được giữ nguyên tỷ lệ
    train_val_df, test_df = train_test_split(df, test_size=test_size, stratify=df['dx'], random_state=42)
    
    # Bước 2: Tách tập Train+Val ra thành tập Train và tập Val.
    # Tỷ lệ val_size là tỷ lệ so với toàn bộ gốc (VD 10%), nên cần quy đổi lại so với train_val_df
    rel_val_size = val_size / (1 - test_size)
    train_df, val_df = train_test_split(train_val_df, test_size=rel_val_size, stratify=train_val_df['dx'], random_state=42)
    
    # Đánh lại chỉ số index để Dataset hoạt động chính xác
    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)
    
    # Lấy ra các transform tương ứng với mô hình
    train_transform = get_transforms(model_type, is_train=True)
    val_test_transform = get_transforms(model_type, is_train=False)
    
    # Tạo các objects Dataset
    train_dataset = SkinCancerDataset(train_df, img_dirs, transform=train_transform)
    val_dataset = SkinCancerDataset(val_df, img_dirs, transform=val_test_transform)
    test_dataset = SkinCancerDataset(test_df, img_dirs, transform=val_test_transform)
    
    # Tạo các DataLoaders để sinh ra các Batch trong quá trình huấn luyện
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class Dinov2LinearProbe(nn.Module):
    """
    Mô hình ứng dụng cơ chế Linear Probing trên đặc trưng của DINOv2.
    - Linear Probing: Đóng băng toàn bộ mạng lớn, chỉ huấn luyện một lớp tuyến tính cuối cùng.
    """
    def __init__(self, num_classes=7, freeze_backbone=True):
        super().__init__()
        # Tải bộ trọng số pre-trained DINOv2 (ViT-Base, patch size 14) từ PyTorch Hub của Meta
        self.backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14')
        
        # Đóng băng (Freeze) các lớp của DINOv2 để không cập nhật trọng số trong lúc huấn luyện
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
                
        # DINOv2 ViT-B/14 tạo ra vector đặc trưng có chiều dài 768
        embed_dim = self.backbone.embed_dim
        # Khởi tạo lớp Linear duy nhất đóng vai trò làm bộ phân loại (Classifier)
        self.head = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x):
        # Trích xuất đặc trưng hình ảnh bằng DINOv2
        features = self.backbone(x)
        # Đưa vector đặc trưng qua lớp Linear để phân loại
        return self.head(features)

class ResNet50Baseline(nn.Module):
    """
    Mô hình tham chiếu (Baseline) sử dụng kiến trúc ResNet50.
    Mô hình này sẽ được Fine-tune toàn bộ có giám sát.
    """
    def __init__(self, num_classes=7, pretrained=True):
        super().__init__()
        # Tải mô hình ResNet50. Sử dụng trọng số có sẵn (ImageNet) nếu pretrained=True
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        self.backbone = models.resnet50(weights=weights)
        
        # Lấy kích thước đầu vào của lớp Fully Connected cuối cùng trong ResNet50
        num_ftrs = self.backbone.fc.in_features
        # Thay thế lớp này bằng một lớp mới phù hợp với số lượng class của SkinCancerMNIST (7)
        self.backbone.fc = nn.Linear(num_ftrs, num_classes)
        
    def forward(self, x):
        return self.backbone(x)

class ClipLinearProbe(nn.Module):
    """
    Mô hình ứng dụng Linear Probing trên Image Encoder của CLIP (OpenCLIP).
    """
    def __init__(self, num_classes=7, freeze_backbone=True):
        super().__init__()
        import open_clip
        # Tải Image Encoder của CLIP với kiến trúc ViT-B/32, được huấn luyện trên tập LAION-2B
        model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
        self.backbone = model.visual # Chỉ lấy phần Visual (Image Encoder), bỏ phần Text Encoder
        
        # Tương tự như DINOv2, đóng băng mạng Backbone
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
                
        # Lấy kích thước đầu ra của CLIP (thường là 512)
        embed_dim = self.backbone.output_dim
        self.head = nn.Linear(embed_dim, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)


In [ ]:
import os
import argparse
import json
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score




def train_epoch(model, dataloader, criterion, optimizer, device):
    """
    Thực hiện 1 Epoch (vòng lặp) huấn luyện trên tập Train.
    """
    model.train() # Chuyển mô hình sang chế độ huấn luyện (kích hoạt Dropout, BatchNorm)
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    # Dùng tqdm để hiển thị thanh tiến trình
    pbar = tqdm(dataloader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad() # Xóa bộ nhớ gradient cũ
        outputs = model(images) # Lan truyền tiến (Forward pass)
        loss = criterion(outputs, labels) # Tính toán Loss
        loss.backward() # Lan truyền ngược (Backward pass) để tính gradient
        optimizer.step() # Cập nhật trọng số
        
        running_loss += loss.item() * images.size(0)
        
        # Dự đoán nhãn (class có xác suất cao nhất)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        
        # Cập nhật thanh tiến trình hiển thị Loss hiện tại
        pbar.set_postfix({"loss": loss.item()})
        
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    """
    Đánh giá mô hình trên tập Validation hoặc Test.
    """
    model.eval() # Chuyển mô hình sang chế độ đánh giá
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    # torch.no_grad() giúp tiết kiệm bộ nhớ, vì lúc này không cần tính gradient
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Evaluating")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    # Dùng macro/weighted f1-score vì tập dữ liệu y tế này mất cân bằng (imbalanced)
    epoch_f1 = f1_score(all_labels, all_preds, average='weighted')
    return epoch_loss, epoch_acc, epoch_f1

def main(args):
    # Cấu hình thiết bị (GPU nếu có, ngược lại dùng CPU)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Sử dụng thiết bị: {device}")
    
    # 1. Chuẩn bị Dữ liệu (Data Pipeline)
    base_dir = "data/SkinCancer"
    csv_path = os.path.join(base_dir, "HAM10000_metadata.csv")
    img_dirs = [os.path.join(base_dir, "HAM10000_images_part_1"), os.path.join(base_dir, "HAM10000_images_part_2")]
    
    print(f"Đang tải dữ liệu cho mô hình: {args.model}")
    train_loader, val_loader, test_loader, num_classes = get_dataloaders(
        csv_path, img_dirs, model_type=args.model, batch_size=args.batch_size, num_workers=4
    )
    
    # 2. Khởi tạo Mô hình
    if args.model == "dinov2":
        model = Dinov2LinearProbe(num_classes=num_classes).to(device)
    elif args.model == "resnet50":
        model = ResNet50Baseline(num_classes=num_classes).to(device)
    elif args.model == "clip":
        model = ClipLinearProbe(num_classes=num_classes).to(device)
    else:
        raise ValueError(f"Không nhận diện được mô hình: {args.model}")
        
    # 3. Thiết lập thông số Huấn luyện
    criterion = nn.CrossEntropyLoss()
    # model.parameters() với DINOv2 và CLIP thì chỉ có trọng số của head là có requires_grad=True
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    
    # Biến theo dõi kết quả để sau này vẽ đồ thị
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }
    
    best_val_acc = 0.0
    os.makedirs("logs", exist_ok=True)
    os.makedirs("weights", exist_ok=True)
    
    # 4. Vòng lặp Huấn luyện chính (Training Loop)
    print(f"Bắt đầu huấn luyện với {args.epochs} epochs...")
    for epoch in range(args.epochs):
        print(f"\nEpoch {epoch+1}/{args.epochs}")
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _ = evaluate(model, val_loader, criterion, device)
        
        # Lưu kết quả
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")
        
        # Chỉ lưu mô hình khi độ chính xác trên tập Validation tăng lên (Early stopping type)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"weights/{args.model}_best.pth")
            
    # 5. Đánh giá cuối cùng trên tập Test
    print("\nTải lại trọng số tốt nhất để chạy đánh giá trên tập Test...")
    model.load_state_dict(torch.load(f"weights/{args.model}_best.pth"))
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion, device)
    
    print(f"Accuracy trên tập Test: {test_acc:.4f}")
    print(f"F1-Score trên tập Test: {test_f1:.4f}")
    
    history["test_acc"] = test_acc
    history["test_f1"] = test_f1
    
    # Xuất lịch sử ra file JSON
    with open(f"logs/{args.model}_history.json", "w") as f:
        json.dump(history, f)

# ==========================================
# CẤU HÌNH HUẤN LUYỆN
# ==========================================
class Args: pass
args = Args()
args.model = "dinov2" # Có thể thay đổi thành "resnet50" hoặc "clip" để so sánh
args.batch_size = 32
args.epochs = 10
args.lr = 1e-3

# Chạy vòng lặp huấn luyện chính
main(args)


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

def load_history(model_name):
    """
    Hàm hỗ trợ đọc file JSON lịch sử kết quả của từng mô hình.
    """
    path = f"logs/{model_name}_history.json"
    if os.path.exists(path):
        with open(path, "r") as f:
            return json.load(f)
    return None

def plot_convergence():
    """
    Hàm vẽ biểu đồ đường (Line chart) để quan sát tốc độ hội tụ của mô hình (Dựa trên Validation Loss).
    Giúp chứng minh nhận định DINOv2 hội tụ rất nhanh nhờ đặc trưng có sẵn.
    """
    models = ["resnet50", "clip", "dinov2"]
    colors = {"resnet50": "blue", "clip": "green", "dinov2": "red"}
    labels = {"resnet50": "ResNet50 (Supervised)", "clip": "CLIP (Zero-shot/Linear)", "dinov2": "DINOv2 (Linear Probe)"}
    
    plt.figure(figsize=(10, 6))
    
    # Đọc kết quả và vẽ lên 1 biểu đồ duy nhất
    for m in models:
        hist = load_history(m)
        if hist:
            val_loss = hist["val_loss"]
            plt.plot(range(1, len(val_loss)+1), val_loss, marker='o', color=colors[m], label=labels[m])
            
    plt.title("Đồ thị hội tụ (Validation Loss)", fontsize=14)
    plt.xlabel("Epochs", fontsize=12)
    plt.ylabel("Validation Loss", fontsize=12)
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    
    # Lưu ra thư mục report để LaTeX tự động cập nhật
    plt.savefig("report/figures/convergence.png", dpi=300, bbox_inches='tight')
    print("Saved convergence.png")

def plot_bar_metrics():
    """
    Hàm vẽ biểu đồ cột (Bar chart) so sánh trực tiếp hiệu năng cuối cùng trên tập Test
    dựa trên 2 tiêu chí cốt lõi: Accuracy và F1-Score.
    """
    models = ["resnet50", "clip", "dinov2"]
    labels = ["ResNet50\n(Supervised)", "CLIP\n(Linear)", "DINOv2\n(Linear)"]
    
    accs = []
    f1s = []
    valid_labels = []
    
    # Lọc ra những mô hình đã có kết quả
    for m, l in zip(models, labels):
        hist = load_history(m)
        if hist:
            accs.append(hist["test_acc"])
            f1s.append(hist["test_f1"])
            valid_labels.append(l)
            
    if not accs:
        print("Chưa có dữ liệu nào để vẽ biểu đồ cột.")
        return
        
    x = np.arange(len(valid_labels))
    width = 0.35 # Độ rộng của cột
    
    fig, ax = plt.subplots(figsize=(8, 6))
    # Vẽ cột Accuracy
    rects1 = ax.bar(x - width/2, accs, width, label='Accuracy', color='skyblue')
    # Vẽ cột F1-Score
    rects2 = ax.bar(x + width/2, f1s, width, label='F1-Score', color='lightcoral')
    
    ax.set_ylabel('Scores', fontsize=12)
    ax.set_title('So sánh hiệu năng các mô hình trên SkinCancerMNIST', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels(valid_labels, fontsize=11)
    ax.legend(loc='lower right')
    ax.set_ylim([0, 1.1]) # Chặn trục Y từ 0 đến 1.1 để đồ thị thoáng
    
    # Hàm con tự động hiển thị số (giá trị) trên đỉnh từng cột
    def autolabel(rects):
        for rect in rects:
            height = rect.get_height()
            ax.annotate(f'{height:.3f}',
                        xy=(rect.get_x() + rect.get_width() / 2, height),
                        xytext=(0, 3),  # Đẩy lên 3 points
                        textcoords="offset points",
                        ha='center', va='bottom', fontsize=10)
    
    autolabel(rects1)
    autolabel(rects2)
    
    fig.tight_layout()
    # Lưu ra thư mục report để LaTeX tự động cập nhật
    plt.savefig("report/figures/performance_bar.png", dpi=300)
    print("Saved performance_bar.png")

# ==========================================
# TRỰC QUAN HÓA KẾT QUẢ ĐỒ THỊ
# ==========================================
import matplotlib.pyplot as plt
import os
os.makedirs("report/figures", exist_ok=True)

# 1. Vẽ đồ thị tốc độ hội tụ
plot_convergence()

# 2. Vẽ đồ thị cột so sánh hiệu năng
plot_bar_metrics()

# Hiển thị trực tiếp ngay trong Jupyter Notebook
plt.show()
